# 12. Inverse EXERKINEMAP & Closed-Loop Optimization
This notebook executes the inverse design workflow, generative sequence sampling via ProGen2, and closed-loop loss minimization ($\mathcal{L}_{INV}$) to optimize candidate exercise-responsive exerkine sequences against target pathway activation states ($A_P^*0$).

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT_ROOT = Path('.').resolve().parents[0]
RESULTS_DIR = PROJECT_ROOT / 'results' / 'optimization'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {DEVICE}')

## Phase 15 & 16: Conditional Sequence Generation
Sampling novel protein candidate sequences ($\hat{\mathbf{a}}^P \sim P_{PLM}(\mathbf{a}^P \mid \mathbf{z}^P)$) conditioned on secretory motifs.

In [ ]:
model_name = 'Salesforce/progen2-small'
print(f'Loading generative model: {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(DEVICE)
model.eval()

# Conditioning prompt motif
prompt = 'MKWVTFISLLFLFSSAYSRV'
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        inputs.input_ids,
        max_new_tokens=45,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_seq = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f'Generated Candidate Sequence: {generated_seq}')

## Phase 17: Closed-Loop Optimization Loop
Iteratively minimizing $\mathcal{L}_{INV} = d(\hat{A}_P, A_P^*)$ until convergence.

In [ ]:
def forward_model_simulation(seq):
    np.random.seed(len(seq))
    return np.random.uniform(0.1, 1.0, size=5)

target_a_p = np.array([0.95, 0.90, 0.85, 0.92, 0.88])
current_seq = generated_seq
epsilon = 0.08
max_iter = 5

for iteration in range(1, max_iter + 1):
    a_p_hat = forward_model_simulation(current_seq)
    loss = float(np.mean((a_p_hat - target_a_p) ** 2))
    print(f'Iteration {iteration} | L_INV Loss: {loss:.4f}')
    
    if loss < epsilon:
        print('Convergence criterion satisfied!')
        break
        
    # Simulated sequence optimization mutation
    mutation_idx = np.random.randint(5, len(current_seq))
    amino_acids = 'VLIFAWMCQEDRKHSTY'
    current_seq = current_seq[:mutation_idx] + amino_acids[np.random.randint(0, len(amino_acids))] + current_seq[mutation_idx+1:]

print('Closed-loop optimization pipeline complete.')